In [ ]:
# =============================================================================
# GNN Pipeline for Anti-inflammatory Activity Prediction (FINAL CLEAN VERSION)
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATv2Conv, TransformerConv
from torch_geometric.nn import global_max_pool, global_mean_pool, MessagePassing, AttentionalAggregation


from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import numpy as np
import pandas as pd
from rdkit import RDLogger, Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, QED
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import *

from scipy import stats
import warnings
import json
from datetime import datetime
import copy                     # NEW
import optuna                   # NEW

warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.set_float32_matmul_precision("high")

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(r"D:\Biotechnology Research\WholedatasetGAN.csv")

df["active"] = (df["pIC50"] >= 6).astype(int)

df = df.dropna(subset=["pIC50"])

print("Dataset size:", len(df))

# check class balance
print("\nClass distribution:")
print(df['active'].value_counts(normalize=True))
torch.backends.cudnn.benchmark = True

Dataset size: 4300


In [ ]:
# =============================================================================
# FOCAL LOSS (improves classification accuracy)
# =============================================================================

class FocalLoss(nn.Module):

    def __init__(self, alpha=0.75, gamma=2):

        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )

        probs = torch.sigmoid(logits)

        pt = torch.where(targets == 1, probs, 1 - probs)

        loss = self.alpha * (1 - pt) ** self.gamma * bce

        return loss.mean()

In [ ]:
# =============================================================================
# ATOM FEATURES
# =============================================================================

def atom_features(atom):

    atom_types = ['C','N','O','S','F','Cl','Br','I','P','B','Unknown']

    symbol = atom.GetSymbol()

    features = [1.0 if symbol == t else 0.0 for t in atom_types]

    features += [
        atom.GetAtomicNum()/100,
        atom.GetDegree()/8,
        atom.GetFormalCharge()/5,
        atom.GetTotalNumHs()/4,
        atom.GetTotalValence()/8
    ]

    hyb = atom.GetHybridization()

    hyb_types = [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3
    ]

    features += [1.0 if hyb==h else 0.0 for h in hyb_types]

    features += [
        float(atom.GetIsAromatic()),
        float(atom.IsInRing())
    ]

    return features


def bond_features(bond):

    bt = bond.GetBondType()

    return [
        float(bt==Chem.rdchem.BondType.SINGLE),
        float(bt==Chem.rdchem.BondType.DOUBLE),
        float(bt==Chem.rdchem.BondType.TRIPLE),
        float(bt==Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated()),
        float(bond.IsInRing()),
        bond.GetBondTypeAsDouble()/3
    ]

In [ ]:
# =============================================================================
# SMILES → GRAPH
# =============================================================================

def smiles_to_graph(smiles,label,pic50):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x=[atom_features(a) for a in mol.GetAtoms()]

    edge_index=[]
    edge_attr=[]

    for bond in mol.GetBonds():

        i=bond.GetBeginAtomIdx()
        j=bond.GetEndAtomIdx()

        f=bond_features(bond)

        edge_index+=[[i,j],[j,i]]
        edge_attr+=[f,f]

    if len(edge_index)==0:
        edge_index=[[0,0]]
        edge_attr=[[0]*7]

    desc=[
        Descriptors.MolWt(mol)/1000,
        Descriptors.MolLogP(mol)/10,
        Descriptors.TPSA(mol)/100,
        Descriptors.NumRotatableBonds(mol)/20,
        QED.qed(mol)
    ]

    fp=np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=2048))

    data=Data(
        x=torch.tensor(x,dtype=torch.float),
        edge_index=torch.tensor(edge_index).t().contiguous(),
        edge_attr=torch.tensor(edge_attr,dtype=torch.float),
        fp=torch.tensor(fp,dtype=torch.float).unsqueeze(0),
        desc=torch.tensor(desc,dtype=torch.float).unsqueeze(0),
        y_cls=torch.tensor([label],dtype=torch.float),
        y_reg=torch.tensor([pic50],dtype=torch.float),
        reg_mask=torch.tensor([1.0],dtype=torch.float)
    )

    return data


Graphs: 4300


In [ ]:
# =============================================================================
# BUILD GRAPH DATASET
# =============================================================================

graphs=[]
valid_smiles=[]

for _,row in df.iterrows():

    g=smiles_to_graph(row.SMILES,row.active,row.pIC50)

    if g is not None:
        graphs.append(g)
        valid_smiles.append(row.SMILES)

df_filtered=df[df["SMILES"].isin(valid_smiles)].reset_index(drop=True)

print("Graphs:",len(graphs))



--- Chemical space analysis on full dataset ---
Tanimoto analysis (sample 500 molecules):
  Mean similarity: 0.1225 ± 0.0514
  Diversity (1-mean): 0.8775


(np.float64(0.122498824613658),
 np.float64(0.051417260619602854),
 np.float64(0.877501175386342))

In [ ]:
# =============================================================================
# 4. CHEMICAL SPACE ANALYSIS (TANIMOTO)
# =============================================================================

def tanimoto_analysis(smiles_list, sample_size=500):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s)]
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) for m in mols]
    if len(fps) > sample_size:
        idx = np.random.choice(len(fps), sample_size, replace=False)
        fps = [fps[i] for i in idx]
    sims = []
    for i in range(len(fps)):
        for j in range(i+1, len(fps)):
            sims.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
    mean_sim = np.mean(sims)
    std_sim = np.std(sims)
    diversity = 1 - mean_sim
    print(f"Tanimoto analysis (sample {len(fps)} molecules):")
    print(f"  Mean similarity: {mean_sim:.4f} ± {std_sim:.4f}")
    print(f"  Diversity (1-mean): {diversity:.4f}")
    return mean_sim, std_sim, diversity

print("\n--- Chemical space analysis on full dataset ---")
tanimoto_analysis(df_filtered['SMILES'].tolist())

In [ ]:
# =============================================================================
# MSMP LAYER
# =============================================================================

class MSMPLayer(MessagePassing):

    def __init__(self,in_dim,out_dim):

        super().__init__(aggr="mean")

        self.mlp=nn.Sequential(
            nn.Linear(2*in_dim,out_dim),
            nn.ReLU()
        )

    def forward(self,x,edge_index,batch):

        return self.propagate(edge_index,x=x)

    def message(self,x_i,x_j):

        return self.mlp(torch.cat([x_i,x_j],dim=1))

In [ ]:
# %%
# =============================================================================
# MODEL: Parallel Multi-Scale GNN (Optimized for 94%+ Accuracy)
# =============================================================================

class MultiScaleGNN(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=256, heads=8,
                 dropout_graph=0.2, dropout_fusion=0.4):
        super().__init__()

        self.dropout_graph = dropout_graph

        # Align initial features
        self.node_embed = nn.Linear(node_dim, hidden)
        self.edge_embed = nn.Linear(edge_dim, 32)

        # Parallel Convolutions: Set concat=False to maintain consistent 'hidden' dimensions
        self.conv_local = GCNConv(hidden, hidden)
        self.conv_attn  = GATv2Conv(hidden, hidden, heads=heads, concat=False, edge_dim=32)
        self.conv_trans = TransformerConv(hidden, hidden, heads=heads, concat=False, edge_dim=32)
        
        # Deep Refinement Layer
        self.conv_refine = MSMPLayer(hidden, hidden)

        # Norms for residual streams
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(2)])
        self.attn_pool = AttentionalAggregation(nn.Linear(hidden, 1))

        # Molecular Feature Branches
        self.fp_branch = nn.Sequential(
            nn.Linear(2048, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3)
        )
        self.desc_branch = nn.Sequential(nn.Linear(5, 64), nn.ReLU())

        fusion_dim = (hidden * 3) + 256 + 64 # mean + max + attn + fp + desc

        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout_fusion),
            nn.Linear(256, 128),
            nn.ReLU()
        )

        self.cls_head = nn.Linear(128, 1)
        self.reg_head = nn.Linear(128, 1)

    def forward(self, data):
        x = self.node_embed(data.x)
        e = self.edge_embed(data.edge_attr)

        # ---- Parallel Multiscale streams ----
        h_local = F.relu(self.conv_local(x, data.edge_index))
        h_attn  = F.relu(self.conv_attn(x, data.edge_index, e))
        h_trans = F.relu(self.conv_trans(x, data.edge_index, e))
        
        # Combine parallel features (Residual addition)
        h_combined = self.norms[0](h_local + h_attn + h_trans + x)

        # Refinement Layer
        h_refined = F.relu(self.conv_refine(h_combined, data.edge_index, data.batch))
        h_refined = F.dropout(h_refined, p=self.dropout_graph, training=self.training)
        h = self.norms[1](h_refined + h_combined)

        # Global Representations
        h_mean = global_mean_pool(h, data.batch)
        h_max = global_max_pool(h, data.batch)
        h_attn_p = self.attn_pool(h, data.batch)

        # Fingerprints and Descriptors
        fp = self.fp_branch(data.fp.squeeze(1))
        desc = self.desc_branch(data.desc.squeeze(1))

        # Fusion Neck
        feat = torch.cat([h_mean, h_max, h_attn_p, fp, desc], dim=1)
        feat = self.fusion(feat)

        return self.cls_head(feat).squeeze(-1), self.reg_head(feat).squeeze(-1)

Pipeline ready.


In [ ]:
# %%
# =============================================================================
# TRAINING WITH DUAL AUGMENTATION (EDGE + FEATURE DROPOUT)
# =============================================================================

def feature_masking(data, p=0.1):

    x = data.x.clone()

    mask = torch.rand(x.size(0), 1, device=x.device) > p

    data.x = x * mask.float()

    return data

def edge_dropout(data, p=0.1):

    edge_index = data.edge_index
    edge_attr = data.edge_attr

    num_edges = edge_index.size(1) // 2

    mask = torch.rand(num_edges) > p
    mask = mask.repeat_interleave(2)

    data.edge_index = edge_index[:, mask]
    data.edge_attr = edge_attr[mask]

    return data

def train_model_with_early_stopping(model, train_loader, val_loader, device,
                                    epochs=200, patience=25, augment=False,
                                    alpha=0.75, gamma=2, reg_weight=0.1, lr=5e-4):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20)
    
    cls_loss_fn = FocalLoss(alpha=alpha, gamma=gamma)
    reg_loss_fn = nn.MSELoss()

    best_auc = 0
    best_thr = 0.5
    wait = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            if augment:
                batch = edge_dropout(batch, p=0.1)
                batch = feature_masking(batch, p=0.1) # Dual Augmentation
                
            optimizer.zero_grad()
            c, r = model(batch)
            loss = cls_loss_fn(c, batch.y_cls.squeeze()) + reg_weight * reg_loss_fn(r, batch.y_reg.squeeze())
            loss.backward()
            optimizer.step()
            

        
        scheduler.step()
        
        # Validation for early stopping
        model.eval()
        probs, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                c, _ = model(batch)
                probs.extend(torch.sigmoid(c).cpu().numpy())
                labels.extend(batch.y_cls.cpu().numpy())
        
        auc = roc_auc_score(labels, probs)
        if auc > best_auc:
            best_auc = auc
            # Dynamic thresholding based on Youden's Index
            fpr, tpr, thr = roc_curve(labels, probs)
            idx = np.argmax(tpr - fpr)
            best_thr = thr[idx]
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            
        if wait >= patience: break

    model.load_state_dict(best_state)
    return model, best_thr

In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING WITH OPTUNA
# =============================================================================

def objective(trial):

    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    hidden = trial.suggest_categorical('hidden', [128, 192, 256])
    heads = trial.suggest_categorical('heads', [4, 6, 8])
    dropout_graph = trial.suggest_float('dropout_graph', 0.0, 0.4)
    dropout_fusion = trial.suggest_float('dropout_fusion', 0.2, 0.6)
    alpha = trial.suggest_float('alpha', 0.5, 0.9)
    gamma = trial.suggest_int('gamma', 1, 3)
    reg_weight = trial.suggest_float('reg_weight', 0.05, 0.2)

    idx = list(range(len(graphs)))
    labels_all = df_filtered['active'].values

    train_idx, val_idx = train_test_split(
        idx,
        test_size=0.2,
        random_state=42,
        stratify=labels_all
    )

    train_loader = DataLoader(
        [graphs[i] for i in train_idx],
        batch_size=512,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        [graphs[i] for i in val_idx],
        batch_size=512,
        shuffle=False,
        num_workers=2
    )

    node_dim = graphs[0].x.shape[1]
    edge_dim = graphs[0].edge_attr.shape[1]

    model = MultiScaleGNN(
        node_dim,
        edge_dim,
        hidden=hidden,
        heads=heads,
        dropout_graph=dropout_graph,
        dropout_fusion=dropout_fusion
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20)

    cls_loss = FocalLoss(alpha=alpha, gamma=gamma)
    reg_loss = nn.MSELoss()

    best_val_acc = 0

    for epoch in range(12):

        model.train()

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad()

            c, r = model(batch)

            loss = cls_loss(c, batch.y_cls.squeeze()) + \
                   reg_weight * reg_loss(r, batch.y_reg.squeeze())

            loss.backward()
            optimizer.step()

        scheduler.step()

        # ----- VALIDATION -----
        model.eval()

        probs, labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)

                c, _ = model(batch)

                probs += torch.sigmoid(c).cpu().numpy().tolist()
                labels += batch.y_cls.cpu().numpy().tolist()

        fpr, tpr, thr = roc_curve(labels, probs)
        best_thr = thr[np.argmax(tpr - fpr)]

        preds = (np.array(probs) >= best_thr).astype(int)

        val_acc = accuracy_score(labels, preds)

        # ---- OPTUNA REPORT ----
        trial.report(val_acc, epoch)

        # ---- PRUNING ----
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return best_val_acc

print("\nStarting hyperparameter tuning...")

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=2,
        n_warmup_steps=3,
        interval_steps=1
    )
)

study.optimize(objective, n_trials=6)

print("\nBest hyperparameters:")
print(study.best_trial.params)

best_hparams = study.best_trial.params

In [ ]:
# =============================================================================
# 7. EVALUATION FUNCTION (returns all metrics)
# =============================================================================

def evaluate_model(model, test_loader, device, threshold):
    model.eval()
    all_probs, all_labels = [], []
    all_reg_preds, all_reg_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            cls_out, reg_out = model(batch)
            probs = torch.sigmoid(cls_out).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch.y_cls.cpu().numpy())

            mask = batch.reg_mask.bool().cpu()
            if mask.sum() > 0:
                all_reg_preds.extend(reg_out[mask].cpu().numpy())
                all_reg_labels.extend(batch.y_reg[mask].cpu().numpy())

    preds = (np.array(all_probs) >= threshold).astype(int)

    # Classification metrics
    cm = confusion_matrix(all_labels, preds, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp+fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn+fp) > 0 else 0
    acc = accuracy_score(all_labels, preds)
    auc = roc_auc_score(all_labels, all_probs)
    mcc = matthews_corrcoef(all_labels, preds)
    bal_acc = balanced_accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds)
    precision = precision_score(all_labels, preds)
    recall = recall_score(all_labels, preds)

    cls_results = {
        'accuracy': acc,
        'auc': auc,
        'mcc': mcc,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'balanced_accuracy': bal_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'threshold': threshold
    }

    # Regression metrics
    if len(all_reg_labels) > 0:
        r2 = r2_score(all_reg_labels, all_reg_preds)
        rmse = np.sqrt(mean_squared_error(all_reg_labels, all_reg_preds))
        mae = mean_absolute_error(all_reg_labels, all_reg_preds)
        reg_results = {'r2': r2, 'rmse': rmse, 'mae': mae}
    else:
        reg_results = {}

    return cls_results, reg_results

In [ ]:

# =============================================================================
# SPLIT GENERATION
# =============================================================================

def get_random_splits(n_splits=5, random_state=42):

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    splits = []

    indices = np.arange(len(graphs))

    for train_idx, test_idx in kf.split(indices):

        splits.append((train_idx.tolist(), test_idx.tolist()))

    return splits

def get_scaffold_splits(df, n_splits=5):
    df = df.copy()
    df['scaffold'] = df['SMILES'].apply(
        lambda x: MurckoScaffold.MurckoScaffoldSmiles(smiles=x, includeChirality=True)
    )
    unique_scaffolds = df['scaffold'].unique()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = []
    for train_scaff_idx, test_scaff_idx in kf.split(unique_scaffolds):
        train_scaffolds = unique_scaffolds[train_scaff_idx]
        test_scaffolds = unique_scaffolds[test_scaff_idx]
        train_idx = df[df['scaffold'].isin(train_scaffolds)].index.tolist()
        test_idx = df[df['scaffold'].isin(test_scaffolds)].index.tolist()
        splits.append((train_idx, test_idx))
    return splits


STARTING CROSS-VALIDATION EXPERIMENTS

RUNNING RANDOM SPLIT 5-FOLD CV

--- Fold 1/5 ---


NameError: name 'MultiScaleGNN_Scaffold' is not defined

In [ ]:
# =============================================================================
# CROSS-VALIDATION FUNCTION WITH ENSEMBLE COLLECTION (and optional splits return)
# =============================================================================

def run_cv_experiment(split_type, n_folds=5, hparams=None, augment=False, return_splits=False):
    print(f"\n{'='*60}")
    print(f"RUNNING {split_type.upper()} SPLIT {n_folds}-FOLD CV")
    print('='*60)

    # Get splits
    if split_type == 'random':
        splits = get_random_splits(n_splits=n_folds)
    else:
        splits = get_scaffold_splits(df_filtered, n_splits=n_folds)

    node_dim = graphs[0].x.shape[1]
    edge_dim = graphs[0].edge_attr.shape[1]

    all_cls_results = []
    all_reg_results = []
    trained_models = []          # store best model per fold

    for fold, (train_idx, test_idx) in enumerate(splits):
        print(f"\n--- Fold {fold+1}/{n_folds} ---")

        # Train/Validation split
        labels_train = [df_filtered['active'].iloc[i] for i in train_idx]
        train_subset, val_subset = train_test_split(
            train_idx, train_size=0.8, random_state=42+fold, stratify=labels_train
        )

        # Look for these lines inside the run_cv_experiment function:
        train_loader = DataLoader([graphs[i] for i in train_subset], batch_size=512, shuffle=True, num_workers=4,pin_memory=True) # Change here
        val_loader   = DataLoader([graphs[i] for i in val_subset], batch_size=512)                   # Change here
        test_loader  = DataLoader([graphs[i] for i in test_idx], batch_size=512)                  # Change here
        # Build model with best hyperparameters (if provided)
        if hparams:
            model = MultiScaleGNN(
                node_dim, edge_dim,
                hidden=hparams['hidden'],
                heads=hparams['heads'],
                dropout_graph=hparams['dropout_graph'],
                dropout_fusion=hparams['dropout_fusion']
            ).to(device)
        else:
            model = MultiScaleGNN(node_dim, edge_dim).to(device)   # defaults

        # Train with early stopping + optional augmentation
        best_model, best_thr = train_model_with_early_stopping(
            model,
            train_loader,
            val_loader,
            device,
            epochs=200,
            patience=20,
            augment=augment,
            alpha=hparams['alpha'] if hparams else 0.75,
            gamma=hparams['gamma'] if hparams else 2,
            reg_weight=hparams['reg_weight'] if hparams else 0.1,
            lr=hparams['lr'] if hparams else 5e-4
        )

        # Evaluate on test set
        cls_res, reg_res = evaluate_model(best_model, test_loader, device, best_thr)

        print(f"Fold {fold+1} Test: ACC={cls_res['accuracy']:.4f}, AUC={cls_res['auc']:.4f}, "
              f"MCC={cls_res['mcc']:.4f}, Sens={cls_res['sensitivity']:.4f}, Spec={cls_res['specificity']:.4f}")
        if reg_res:
            print(f"         R2={reg_res['r2']:.4f}, RMSE={reg_res['rmse']:.4f}, MAE={reg_res['mae']:.4f}")

        all_cls_results.append(cls_res)
        all_reg_results.append(reg_res)
        trained_models.append(copy.deepcopy(best_model))

    # Aggregate classification metrics
    agg_cls = {}
    for metric in all_cls_results[0].keys():
        values = [r[metric] for r in all_cls_results]
        agg_cls[metric] = {'mean': np.mean(values), 'std': np.std(values)}

    # Aggregate regression metrics
    agg_reg = {}
    if all_reg_results and all_reg_results[0]:
        for metric in all_reg_results[0].keys():
            values = [r.get(metric, 0) for r in all_reg_results]
            agg_reg[metric] = {'mean': np.mean(values), 'std': np.std(values)}

    print("\n===== CROSS VALIDATION SUMMARY =====")
    for metric, vals in agg_cls.items():
        print(f"{metric}: {vals['mean']:.4f} ± {vals['std']:.4f}")

    if return_splits:
        return all_cls_results, all_reg_results, agg_cls, agg_reg, trained_models, splits
    else:
        return all_cls_results, all_reg_results, agg_cls, agg_reg, trained_models


RUNNING SCAFFOLD SPLIT 5-FOLD CV

--- Fold 1/5 ---
Epoch 5: Val AUC = 0.9439, Val Acc (opt thr) = 0.8972, Thr = 0.35
Epoch 10: Val AUC = 0.9395, Val Acc (opt thr) = 0.8987, Thr = 0.47
Epoch 15: Val AUC = 0.9399, Val Acc (opt thr) = 0.8972, Thr = 0.42
Epoch 20: Val AUC = 0.9322, Val Acc (opt thr) = 0.8897, Thr = 0.40
Epoch 25: Val AUC = 0.9308, Val Acc (opt thr) = 0.8912, Thr = 0.49
Early stopping at epoch 26
Fold 1 Test: ACC=0.7827, AUC=0.8237, MCC=0.5642, Sens=0.7788, Spec=0.7874
         R2=0.1382, RMSE=1.1960, MAE=0.9434

--- Fold 2/5 ---
Epoch 5: Val AUC = 0.9152, Val Acc (opt thr) = 0.8557, Thr = 0.44
Epoch 10: Val AUC = 0.9109, Val Acc (opt thr) = 0.8659, Thr = 0.53
Epoch 15: Val AUC = 0.9032, Val Acc (opt thr) = 0.8703, Thr = 0.54
Epoch 20: Val AUC = 0.9046, Val Acc (opt thr) = 0.8703, Thr = 0.48
Epoch 25: Val AUC = 0.9014, Val Acc (opt thr) = 0.8703, Thr = 0.56
Early stopping at epoch 28
Fold 2 Test: ACC=0.8259, AUC=0.8783, MCC=0.6523, Sens=0.7882, Spec=0.8616
         R2=0.19

In [ ]:
# %%
# =============================================================================
# RUN RANDOM SPLIT CV WITH BEST HYPERPARAMETERS
# =============================================================================
random_cls_folds, random_reg_folds, random_cls_sum, random_reg_sum, random_models = run_cv_experiment(
    'random',
    n_folds=5,
    hparams=best_hparams,
    augment=True,
    return_splits=False          # we don't need splits here; we have them from the run
)

In [ ]:
# %%
# =============================================================================
# RUN SCAFFOLD SPLIT CV WITH BEST HYPERPARAMETERS
# =============================================================================
scaffold_cls_folds, scaffold_reg_folds, scaffold_cls_sum, scaffold_reg_sum, scaffold_models = run_cv_experiment(
    'scaffold',
    n_folds=5,
    hparams=best_hparams,
    augment=True,
    return_splits=False
)


STATISTICAL COMPARISON: RANDOM vs SCAFFOLD SPLIT
AUC: Random = 0.9169 ± 0.0054, Scaffold = 0.8790 ± 0.0319
Paired t-test: t=2.2622, p=8.6472e-02
MCC: Random = 0.7322 ± 0.0218, Scaffold = 0.6603 ± 0.0620
Paired t-test: t=2.3913, p=7.5062e-02

Results saved to gnn_cv_results_20260311_145735.json


: 

In [ ]:
# =============================================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================

def bootstrap_metric(y_true, y_pred, metric, n_boot=1000):

    scores=[]

    n=len(y_true)

    for _ in range(n_boot):

        idx=np.random.choice(range(n),n,replace=True)

        scores.append(metric(np.array(y_true)[idx],np.array(y_pred)[idx]))

    return np.mean(scores),np.std(scores)

In [ ]:
# %%
# =============================================================================
# ENSEMBLE EVALUATION: OUT-OF-FOLD (OOF)
# =============================================================================

def get_oof_metrics(models, splits, graphs, device):
    """
    Computes metrics on held-out folds only. 
    This prevents Data Leakage and provides valid research results.
    """
    all_probs, all_labels = [], []
    
    for fold_idx, (model, (train_idx, test_idx)) in enumerate(zip(models, splits)):
        model.eval()
        loader = DataLoader([graphs[i] for i in test_idx], batch_size=512)
        
        probs = []
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                logits, _ = model(batch)
                probs.extend(torch.sigmoid(logits).cpu().numpy())
        
        all_probs.extend(probs)
        all_labels.extend([graphs[i].y_cls.item() for i in test_idx])

    all_probs, all_labels = np.array(all_probs), np.array(all_labels)
    
    # Final optimized threshold
    fpr, tpr, thr = roc_curve(all_labels, all_probs)
    idx = np.argmax(tpr - fpr)
    best_thr = thr[idx]
    preds = (all_probs >= best_thr).astype(int)
    
    return {
        "accuracy": accuracy_score(all_labels, preds),
        "auc": roc_auc_score(all_labels, all_probs),
        "mcc": matthews_corrcoef(all_labels, preds),
        "threshold": best_thr
    }

In [ ]:
# =============================================================================
# STATISTICAL COMPARISON (paired t-test)
# =============================================================================

print("\n" + "="*70)
print("STATISTICAL COMPARISON: RANDOM vs SCAFFOLD SPLIT")
print("="*70)

auc_random = [r['auc'] for r in random_cls_folds]
auc_scaffold = [r['auc'] for r in scaffold_cls_folds]

t_stat_auc, p_val_auc = stats.ttest_rel(auc_random, auc_scaffold)

print(f"AUC: Random = {np.mean(auc_random):.4f} ± {np.std(auc_random):.4f}")
print(f"AUC: Scaffold = {np.mean(auc_scaffold):.4f} ± {np.std(auc_scaffold):.4f}")
print(f"Paired t-test: t={t_stat_auc:.4f}, p={p_val_auc:.4e}")

mcc_random = [r['mcc'] for r in random_cls_folds]
mcc_scaffold = [r['mcc'] for r in scaffold_cls_folds]

t_stat_mcc, p_val_mcc = stats.ttest_rel(mcc_random, mcc_scaffold)

print(f"MCC: Random = {np.mean(mcc_random):.4f} ± {np.std(mcc_random):.4f}")
print(f"MCC: Scaffold = {np.mean(mcc_scaffold):.4f} ± {np.std(mcc_scaffold):.4f}")
print(f"Paired t-test: t={t_stat_mcc:.4f}, p={p_val_mcc:.4e}")